# Extended Kalman Filter, A Revisit on the Non-Linear State Space Model

In [ ]:
import pandas as pd
import numpy as np

import numpyro
numpyro.set_host_device_count(4)
import jax
import jax.numpy as jnp

from numpyro import distributions as dist
from numpyro.infer import MCMC, NUTS

import matplotlib.pyplot as plt
from jax import random

from wunkui.models.ssp.kalman_1d_ekf import (
    kalman_dk_smoother_1d_ekf,
    kalman_filter_1d_ekf,
    kalman_rts_smoother_1d_ekf,
 )
from wunkui.models.ssp import a_to_lam, plot_states, transform_to_ekf
from wunkui.models.ssp.utils import construct_states_prior

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# ── Hyperparameters ───────────────────────────────────────────────────────────
# False → plain Kalman filter; True → enforce a_obs at sampled windows
USE_DISCLOSURE = True 
# number of disclosure windows to sample
N_PERIODS      = 3    
# consecutive steps per window
N_POINTS       = 10     
# disclose noise variance
DISC_OBS_SIGMA = 0.01

# for EKF positivity constraint
EXPONENT = 0.5


In [ ]:
df = pd.read_csv(
    '../resource/simple/df.csv', parse_dates=['date']
)
# for quick testing
df = df[-365:].reset_index(drop=True)

saturation_df = pd.read_csv(
    '../resource/simple/saturation_df.csv',
)
coefs_df = pd.read_csv(
    '../resource/simple/coefs_df.csv',
)

regressors = saturation_df["regressor"].tolist()
response = df["sales"].values

response_norm = response.mean()
y = np.log(response / response_norm)
y = jnp.array(y)
X = df[regressors].values
sat_arr = saturation_df.set_index("regressor").loc[regressors, "saturation"].values
X = np.log1p(X / sat_arr)

Z = jnp.concatenate([jnp.ones((X.shape[0], 1)), jnp.array(X)], -1)

positivity_idx = jnp.array([0] + [1] * len(regressors))
positivity_idx = positivity_idx.astype(bool)

print("y shape", y.shape)
print("Z shape", Z.shape)
print("X shape", X.shape)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(df["date"], y, label="y")

In [ ]:
n_steps = y.shape[0]
n_states = Z.shape[-1]
ssp_priors_nat = {}

if coefs_df is not None:
    coefs_lookup = coefs_df.set_index("regressor")["coef"]
    # natural-scale true state: intercept linear, channel coefficients in λ-space
    true_state_nat = jnp.array([0.0] + [coefs_lookup[r] for r in regressors])

    if USE_DISCLOSURE:
        obs_priors = construct_states_prior(
            n_steps=n_steps,
            n_states=n_states,
            true_states=true_state_nat,
            regressors=regressors,
            n_periods=N_PERIODS,
            n_points=N_POINTS,
            obs_scale=DISC_OBS_SIGMA,
        )
        ssp_priors_nat.update({
            "a_obs_nat": obs_priors["a_obs"],
            "P_obs_nat": obs_priors["P_obs"],
            "obs_idx": obs_priors["obs_idx"],
        })
        print("n disclosed steps:", len(ssp_priors_nat["obs_idx"]))
    else:
        ssp_priors_nat.update({"a_obs_nat": None, "P_obs_nat": None, "obs_idx": np.array([], dtype=int)})
        print("disclosure disabled")

    print("a_obs_nat:", None if ssp_priors_nat["a_obs_nat"] is None else ssp_priors_nat["a_obs_nat"].shape)
    print("P_obs_nat:", None if ssp_priors_nat["P_obs_nat"] is None else ssp_priors_nat["P_obs_nat"].shape)
else:
    ssp_priors_nat.update({"a_obs_nat": None, "P_obs_nat": None, "obs_idx": np.array([], dtype=int)})
    print("no ground truth state available; disclosure disabled")

In [ ]:
n_regressors = X.shape[-1]

ssp_priors_nat["a0_nat"] = jnp.array([0.0] + [0.1] * n_regressors)
# Tighter natural-scale variance on channels: lognormal moment-match blows
# up Var/μ² ratios, so shrinking 0.1 → 0.01 keeps the a-space prior compact.
ssp_priors_nat["P0_nat"] = jnp.diag(jnp.array([1.0] + [0.01] * n_regressors))
# (level, regressor) process-noise scale priors in the natural scale.
# sigma_q is a standard deviation, not a latent-state mean/variance pair.
ssp_priors_nat["sigma_q_loc_prior_nat"] = np.array([0.05, 0.01])
ssp_priors_nat["sigma_q_scale_prior_nat"] = np.array([0.05, 0.01])
# Truncation bounds in natural scale. For positive states transform_to_ekf
# converts sigma_q with the same local positive scale map used for the
# TruncatedNormal prior parameters; it does not moment-match sigma_q like a0/P0.
ssp_priors_nat["sigma_q_low_prior_nat"] = np.array([1e-5, 1e-5])
ssp_priors_nat["sigma_q_high_prior_nat"] = np.array([0.1, 0.1])

print("----- Natural-scale priors -----")
print("ssp_priors_nat keys:", ssp_priors_nat.keys())
print("a0:", ssp_priors_nat["a0_nat"])
print("P0 diag:", jnp.diag(ssp_priors_nat["P0_nat"]))
print("sigma_q loc / scale:", ssp_priors_nat["sigma_q_loc_prior_nat"], "/", ssp_priors_nat["sigma_q_scale_prior_nat"])
print("sigma_q low / high:", ssp_priors_nat["sigma_q_low_prior_nat"], "/", ssp_priors_nat["sigma_q_high_prior_nat"])

ssp_priors_ekf = transform_to_ekf(ssp_priors_nat, positivity_idx, exponent=EXPONENT)
print("\n----- EKF-space priors -----")
print("ssp_priors_ekf keys:", ssp_priors_ekf.keys())
print("a0:", ssp_priors_ekf["a0"])

print("P0 diag:", jnp.diag(ssp_priors_ekf["P0"]))
print("sigma_q loc / scale:", ssp_priors_ekf["sigma_q_loc_prior"], "/", ssp_priors_ekf["sigma_q_scale_prior"])
print("sigma_q low / high:", ssp_priors_ekf["sigma_q_low_prior"], "/", ssp_priors_ekf["sigma_q_high_prior"])


In [ ]:
a0      = ssp_priors_ekf["a0"]
# EKF expects diagonal variances; collapse the full a-space covariance.
P0      = jnp.diag(ssp_priors_ekf["P0"])
a_obs   = ssp_priors_ekf["a_obs"]
P_obs   = ssp_priors_ekf["P_obs"]
obs_idx = ssp_priors_ekf["obs_idx"]

print("a0 shape:", a0.shape)
print(f"  a0[0] = {a0[0]:.3f}  (intercept, linear space)")
print(
    f"  a0[1] = {a0[1]:.3f}  (channel, a-space → λ₀ = exp({EXPONENT}·a0) = "
    f"{jnp.exp(EXPONENT * a0[1]):.3f})"
)
print("P0:", P0)

In [ ]:
def test_run(ssp_priors, positivity_idx, exponent=EXPONENT):
    sigma_h = jnp.array(0.1)
    # sigma_q_raw = jnp.array([0.01, 0.01])  # [level, shared_regressors]
    sigma_q_raw = ssp_priors["sigma_q_loc_prior"]  # using prior loc as a test value
    sigma_q = jnp.concatenate([sigma_q_raw[:1], jnp.repeat(sigma_q_raw[1:], Z.shape[-1] - 1)])
    print("sigma_q expanded:", sigma_q.shape, sigma_q)

    lp, at, Pt, _, _, _ = kalman_filter_1d_ekf(
        a0=a0,
        P0=P0,
        sigma_h=sigma_h,
        sigma_q=sigma_q,
        y=y,
        Z=Z,
        logp=True,
        exponent=EXPONENT,
        positivity_idx=positivity_idx,
    )

    # recover intensities: intercept stays as-is, channels via exp(EXPONENT·a)
    lam = jnp.where(positivity_idx, jnp.exp(jnp.clip(exponent * at, -10.0, 10.0)), at)

    print("lp:", lp)
    print("at shape:", at.shape)
    print("intercept at[-1,0]:", at[-1, 0], "  ← linear space")
    print(f"channel lam[-1,1]: {lam[-1, 1]}  ← λ = exp({exponent}·a)")

test_run(ssp_priors_ekf, positivity_idx)

In [ ]:
def make_nuts_fn(ssp_priors_ekf, y, Z, positivity_idx, exponent):
    a0    = ssp_priors_ekf["a0"]
    P0    = jnp.diag(ssp_priors_ekf["P0"])
    a_obs = ssp_priors_ekf["a_obs"]
    P_obs = ssp_priors_ekf["P_obs"]
    # NumPyro samples sigma_q in the (level, shared-regressor) parameterisation;
    # collapse the per-state hyperpriors to that two-element form.
    sigma_q_loc_prior   = ssp_priors_ekf["sigma_q_loc_prior"]
    sigma_q_scale_prior = ssp_priors_ekf["sigma_q_scale_prior"]
    sigma_q_low_prior   = ssp_priors_ekf["sigma_q_low_prior"]
    sigma_q_high_prior  = ssp_priors_ekf["sigma_q_high_prior"]
    positivity_idx      = positivity_idx.astype(bool)

    def nuts_fn():
        sigma_h = numpyro.sample(
            "sigma_h",
            dist.TruncatedNormal(0.1, 1.0, high=1.0, low=1e-5),
        )
        sigma_q_raw = numpyro.sample(
            "sigma_q",
            dist.TruncatedNormal(
                sigma_q_loc_prior,
                sigma_q_scale_prior,
                low=sigma_q_low_prior,
                high=sigma_q_high_prior,
            ),
        )
        n_regressors = Z.shape[-1] - 1
        sigma_q = jnp.concatenate([sigma_q_raw[:1], jnp.repeat(sigma_q_raw[1:], n_regressors)])

        lp, at, Pt, vt, Ft, Kt = kalman_filter_1d_ekf(
            a0=a0,
            P0=P0,
            sigma_h=sigma_h,
            sigma_q=sigma_q,
            y=y,
            Z=Z,
            logp=True,
            exponent=exponent,
            positivity_idx=positivity_idx,
            a_obs=a_obs,
            P_obs=P_obs,
        )
        lam = jnp.where(positivity_idx, jnp.exp(jnp.clip(exponent * at, -10.0, 10.0)), at)

        numpyro.factor("lp", lp)
        numpyro.deterministic("at", at)
        numpyro.deterministic("Pt", Pt)
        numpyro.deterministic("lam", lam)
        numpyro.deterministic("mu", jnp.sum(Z * lam, -1))
        numpyro.deterministic("sigma_q_full", sigma_q)

    return nuts_fn

In [ ]:
nuts_fn = make_nuts_fn(ssp_priors_ekf, y, Z, positivity_idx, exponent=EXPONENT)
nuts_kernel = NUTS(nuts_fn)
mcmc = MCMC(
    nuts_kernel,
    num_warmup=400,
    num_samples=400,
    num_chains=4,
)
rng_key = random.PRNGKey(0)
rng_keys = random.split(rng_key, 2)
rng_key = rng_keys[0]
mcmc_rng_key = rng_keys[1]
mcmc.run(mcmc_rng_key)

In [ ]:
from numpyro.diagnostics import summary

# print_summary covers R-hat and n_eff for all sites
mcmc.print_summary()

# detailed per-parameter diagnostics for scalar params
samples = mcmc.get_samples(group_by_chain=True)
diag = summary(samples, prob=0.9)

# flag any R-hat > 1.01 or n_eff < 100
# use max(r_hat) / min(n_eff) for multi-dimensional params (at, lam, mu, etc.)
print("\n--- Convergence flags ---")
for param, stats in diag.items():
    rhat = float(np.asarray(stats["r_hat"]).max())
    neff = float(np.asarray(stats["n_eff"]).min())
    if rhat > 1.01 or neff < 100:
        print(f"  WARNING  {param:<20}  r_hat={rhat:.4f}  n_eff={neff:.1f}")

In [ ]:
posteriors_dict = mcmc.get_samples()
posteriors_dict.keys()

# ── Post-MCMC RTS and D&K smoothers ───────────────────────────────────────
# RTS needs only (at, Pt, sigma_q_full). D&K also needs the EKF innovations,
# innovation variances, and Kalman gains, so rebuild those outside the model
# by vmapping the EKF filter over posterior draws.

at_draws      = posteriors_dict["at"]            # (n_draws, T, n_states)
Pt_draws      = posteriors_dict["Pt"]            # (n_draws, T, n_states)
sigma_h_draws = posteriors_dict["sigma_h"]       # (n_draws,)
sigma_q_draws = posteriors_dict["sigma_q_full"]  # (n_draws, n_states)

def _filter_from_draw(sigma_h_i, sigma_q_i):
    _, at_i, Pt_i, vt_i, Ft_i, Kt_i = kalman_filter_1d_ekf(
        a0=a0,
        P0=P0,
        sigma_h=sigma_h_i,
        sigma_q=sigma_q_i,
        y=y,
        Z=Z,
        logp=True,
        exponent=EXPONENT,
        positivity_idx=positivity_idx,
        a_obs=a_obs,
        P_obs=P_obs,
    )
    return at_i, Pt_i, vt_i, Ft_i, Kt_i

_, _, vt_draws, Ft_draws, Kt_draws = jax.vmap(_filter_from_draw)(
    sigma_h_draws, sigma_q_draws,
 )

at_smooth, Pt_smooth = jax.vmap(
    lambda at_i, Pt_i, sigma_q_i: kalman_rts_smoother_1d_ekf(
        at=at_i,
        Pt=Pt_i,
        sigma_q=sigma_q_i,
    )
 )(at_draws, Pt_draws, sigma_q_draws)

at_smooth_dk = jax.vmap(
    lambda at_i, Pt_i, vt_i, Ft_i, Kt_i, sigma_q_i: kalman_dk_smoother_1d_ekf(
        at=at_i,
        Pt=Pt_i,
        vt=vt_i,
        Ft=Ft_i,
        Kt=Kt_i,
        Z=Z,
        a0=a0,
        P0=P0,
        sigma_q=sigma_q_i,
        exponent=EXPONENT,
        positivity_idx=positivity_idx,
        a_obs=a_obs,
        P_obs=P_obs,
    )
 )(at_draws, Pt_draws, vt_draws, Ft_draws, Kt_draws, sigma_q_draws)

lam_smooth = jnp.where(
    positivity_idx[None, None, :],
    jnp.exp(jnp.clip(EXPONENT * at_smooth, -10.0, 10.0)),
    at_smooth,
 )
lam_smooth_dk = jnp.where(
    positivity_idx[None, None, :],
    jnp.exp(jnp.clip(EXPONENT * at_smooth_dk, -10.0, 10.0)),
    at_smooth_dk,
 )

print(
    f"RTS at_smooth shape: {at_smooth.shape}    DK at_smooth shape: {at_smooth_dk.shape}\n"
    f"max |RTS - DK| in a-space: {float(jnp.max(jnp.abs(at_smooth - at_smooth_dk))):.3e}\n"
    f"max |RTS - DK| in λ-space: {float(jnp.max(jnp.abs(lam_smooth - lam_smooth_dk))):.3e}"
 )

posteriors_dict["at_smooth"] = np.asarray(at_smooth)
posteriors_dict["Pt_smooth"] = np.asarray(Pt_smooth)
posteriors_dict["at_smooth_dk"] = np.asarray(at_smooth_dk)
posteriors_dict["lam_smooth"] = np.asarray(lam_smooth)
posteriors_dict["lam_smooth_dk"] = np.asarray(lam_smooth_dk)
posteriors_dict["mu"] = np.asarray(jnp.einsum("dts,ts->dt", lam_smooth, Z))
posteriors_dict["mu_dk"] = np.asarray(jnp.einsum("dts,ts->dt", lam_smooth_dk, Z))

In [ ]:
state_labels = ["intercept"] + regressors
a_obs = ssp_priors_nat["a_obs_nat"] if USE_DISCLOSURE else None
P_obs = ssp_priors_nat["P_obs_nat"] if USE_DISCLOSURE else None
obs_idx = ssp_priors_nat["obs_idx"] if USE_DISCLOSURE else None

plot_states(
    posteriors_dict,
    df["date"].values,
    state_labels,
    states_key="lam",
    coefs_df=coefs_df,
    obs_idx=obs_idx,
    a_obs=a_obs,
    P_obs=P_obs,
    title=(
        f"Log-normal EKF — λ_t = exp({EXPONENT}·a_t)  [disclosure ON]"
        if USE_DISCLOSURE
        else f"Log-normal EKF — λ_t = exp({EXPONENT}·a_t)  [disclosure OFF]"
    ),
)
plt.show()

plot_states(
    posteriors_dict,
    df["date"].values,
    state_labels,
    states_key="lam_smooth",
    coefs_df=coefs_df,
    obs_idx=obs_idx,
    a_obs=a_obs,
    P_obs=P_obs,
    title=(
        f"Log-normal EKF — RTS smoothed λ_t = exp({EXPONENT}·a_t)  [disclosure ON]"
        if USE_DISCLOSURE
        else f"Log-normal EKF — RTS smoothed λ_t = exp({EXPONENT}·a_t)  [disclosure OFF]"
    ),
)
plt.show()

plot_states(
    posteriors_dict,
    df["date"].values,
    state_labels,
    states_key="lam_smooth_dk",
    coefs_df=coefs_df,
    obs_idx=obs_idx,
    a_obs=a_obs,
    P_obs=P_obs,
    title=(
        f"Log-normal EKF — D&K smoothed λ_t = exp({EXPONENT}·a_t)  [disclosure ON]"
        if USE_DISCLOSURE
        else f"Log-normal EKF — D&K smoothed λ_t = exp({EXPONENT}·a_t)  [disclosure OFF]"
    ),
)
plt.show()

In [ ]:
mu_samples = np.array(posteriors_dict["mu"])
eps_samples = np.random.normal(
    0,
    np.array(posteriors_dict["sigma_h"])[:, None],
    size=mu_samples.shape,
)

# compute p5, p50, p95 for forecast
yhat_lower, yhat_mid, yhat_upper = np.quantile(np.exp(mu_samples + eps_samples) * response_norm, q=[0.05, 0.5, 0.95], axis=0)

n = 100
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(np.arange(n), response[-n:], label='Observed', alpha=0.5, color="orange", s=10)
ax.plot(np.arange(n), yhat_mid[-n:], label='Forecast', linestyle='--', alpha=0.8, color="dodgerblue")
ax.fill_between(np.arange(n), yhat_lower[-n:], yhat_upper[-n:], alpha=0.5, label='95% Prediction Interval', color="dodgerblue")
ax.set_title('State Space Model Forecast')